# Child simulation — automated V&V

Baseline-scenario sim output vs. the child artifact, using only the built-in
`RatioMeasure` classes. See `../README.md` for the measure triage.

Requires a psimulate run made *after* `PersonTimeObserver` became a
`PublicHealthObserver` — older runs have no `person_time_*` dataset and no denominator.

In [ ]:
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

from vivarium.validation import ValidationContext

from lsff_utils import paths

In [ ]:
LOCATION = "nigeria"
VEHICLE = "bouillon"
MODEL_NUMBER = paths.MODEL_NUMBER  # the archived iteration to check; set explicitly to read an older one
RUN = ""  # psimulate run timestamp; blank takes the run archive_last_run.sh published

# The archive is where a finished iteration lives, and the marker copied beside the
# run directories names the run the pipeline published -- so nothing here is pasted in.
results_root = paths.archive_root(paths.CHILD_RESULTS_ROOT, MODEL_NUMBER)
RESULTS_DIR = (
    paths.run_root(results_root, LOCATION, VEHICLE) / RUN
    if RUN
    else paths.latest_run(results_root, LOCATION, VEHICLE)
)

REPORT_PATH = f"child_validation_report_{VEHICLE}_{LOCATION}.html"
print(f"{MODEL_NUMBER} {VEHICLE}/{LOCATION}: {RESULTS_DIR}")

In [ ]:
# psimulate names branch columns after the config-path leaf.
vc = ValidationContext(
    results_dir=RESULTS_DIR,
    scenario_columns=["child_scenario", "maternal_scenario"],
)

## Available simulation outputs

Expect `person_time_population` and the derived `person_time_total` — the loader builds the
latter from the largest `person_time_*` dataset, and it is the denominator for every rate.

In [ ]:
vc.get_sim_outputs()

## Available artifact keys

In [ ]:
vc.get_artifact_keys()

## Comparisons

In [ ]:
BASELINE = {"child_scenario": "baseline", "maternal_scenario": "baseline"}

vc.add_comparison(
    "cause.all_causes.cause_specific_mortality_rate",
    test_source="sim",
    ref_source="artifact",
    test_scenarios=BASELINE,
)

sorted(vc.comparisons)

## Report

In [ ]:
vc.generate_results(output_path=REPORT_PATH)